# Notebook 05 — Embeddings y Transformers

## Objetivo

Comparar modelos más complejos sobre el **subconjunto político**:
- **Parte A:** Embeddings estáticos (GloVe preentrenado + Word2Vec de dominio) + LR/SVM
- **Parte B:** Fine-tuning de DistilBERT con grilla de hiperparámetros

Texto de entrada: `clean_full_text_without_stopwords` (título + cuerpo limpios, alineado con el pipeline de preprocesamiento). Se respeta la decisión de ablación de fuente del Experimento 1.

BERT-base queda fuera del alcance local (CPU); ver ADR-03. Opcionalmente se puede correr la Parte B en **Google Colab con GPU**.

> Los modelos aprenden patrones lingüísticos del dataset; no verifican hechos.

In [ ]:
import warnings
from itertools import product

warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from sklearn.metrics import accuracy_score, fbeta_score, precision_recall_fscore_support
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)

from nlp.embeddings import (
    embedding_text_series,
    load_glove_vectors,
    load_or_compute_document_embeddings,
    load_or_compute_glove_embeddings,
    train_or_load_word2vec,
    tune_and_evaluate_dense_classifiers,
)
from nlp.features import load_source_normalization_decision
from nlp.io import load_splits
from nlp.metrics import compute_metrics, consolidate_results, metrics_to_row
from nlp.paths import *
from nlp.plotting import save_figure, setup_style
from nlp.transformers_data import NewsDataset, prepare_transformer_texts

setup_style()
FIGURE_PREFIX = "05_"

## 1. Datos y decisión de fuente

Mismo criterio que los notebooks 03–04: leer `source_ablation_decision.json` y aplicar `normalize_source_markers` solo si la caída de F2 en validación superó el umbral.

In [ ]:
EMBEDDING_COLS = ["label", "title", "text", "clean_full_text_without_stopwords"]
pol_train, pol_val, pol_test = load_splits("politics", columns=EMBEDDING_COLS)

TEXT_COL = "clean_full_text_without_stopwords"
EMB_DIM = 100
CACHE_PREFIX = "politics"

source_decision = load_source_normalization_decision(SOURCE_ABLATION_DECISION)
normalize_source = source_decision["use_source_normalization"]

if normalize_source:
    print(
        f"Normalizar fuentes en entrada "
        f"(caída F2 val = {source_decision['f2_drop']:.3f} "
        f"≥ {source_decision['threshold']})."
    )
else:
    print(
        f"Texto original en entrada "
        f"(caída F2 val = {source_decision['f2_drop']:.3f} "
        f"< {source_decision['threshold']})."
    )

train_texts = embedding_text_series(pol_train, TEXT_COL, normalize_source=normalize_source)
val_texts = embedding_text_series(pol_val, TEXT_COL, normalize_source=normalize_source)
test_texts = embedding_text_series(pol_test, TEXT_COL, normalize_source=normalize_source)
y_train, y_val, y_test = pol_train["label"], pol_val["label"], pol_test["label"]

## 2. Parte A — Embeddings estáticos (GloVe + Word2Vec)

Representación de documento: **promedio** de vectores de palabra. Clasificador: LR o LinearSVC con `StandardScaler`.

In [ ]:
glove = load_glove_vectors(EMB_DIM)
print(f"GloVe: {len(glove):,} vectores")

w2v_model = train_or_load_word2vec(
    train_texts,
    word2vec_model_path(CACHE_PREFIX, EMB_DIM),
    vector_size=EMB_DIM,
)
w2v_vectors = w2v_model.wv
print(f"Word2Vec: {len(w2v_vectors):,} vectores (entrenado en train politics)")

In [ ]:
X_train_glove = load_or_compute_glove_embeddings(
    train_texts,
    DATA_EMBEDDINGS / f"{CACHE_PREFIX}_train_glove{EMB_DIM}d.npz",
    glove,
    EMB_DIM,
)
X_val_glove = load_or_compute_glove_embeddings(
    val_texts,
    DATA_EMBEDDINGS / f"{CACHE_PREFIX}_val_glove{EMB_DIM}d.npz",
    glove,
    EMB_DIM,
)
X_test_glove = load_or_compute_glove_embeddings(
    test_texts,
    DATA_EMBEDDINGS / f"{CACHE_PREFIX}_test_glove{EMB_DIM}d.npz",
    glove,
    EMB_DIM,
)

X_train_w2v = load_or_compute_document_embeddings(
    train_texts,
    DATA_EMBEDDINGS / f"{CACHE_PREFIX}_train_w2v{EMB_DIM}d.npz",
    w2v_vectors,
    EMB_DIM,
    tag="word2vec",
)
X_val_w2v = load_or_compute_document_embeddings(
    val_texts,
    DATA_EMBEDDINGS / f"{CACHE_PREFIX}_val_w2v{EMB_DIM}d.npz",
    w2v_vectors,
    EMB_DIM,
    tag="word2vec",
)
X_test_w2v = load_or_compute_document_embeddings(
    test_texts,
    DATA_EMBEDDINGS / f"{CACHE_PREFIX}_test_w2v{EMB_DIM}d.npz",
    w2v_vectors,
    EMB_DIM,
    tag="word2vec",
)

In [ ]:
embedding_results = []
embedding_results.extend(
    tune_and_evaluate_dense_classifiers(
        X_train_glove,
        y_train,
        X_val_glove,
        y_val,
        X_test_glove,
        y_test,
        representation="glove_avg",
        use_source_normalization=normalize_source,
    )
)
embedding_results.extend(
    tune_and_evaluate_dense_classifiers(
        X_train_w2v,
        y_train,
        X_val_w2v,
        y_val,
        X_test_w2v,
        y_test,
        representation="word2vec_avg",
        use_source_normalization=normalize_source,
    )
)

embedding_df = pd.DataFrame(embedding_results)
embedding_df.to_csv(RESULTS_METRICS / "embedding_results.csv", index=False)
embedding_df

In [ ]:
plot_emb = embedding_df.copy()
plot_emb["label"] = plot_emb["representation"] + " / " + plot_emb["model"]
plot_emb = plot_emb.sort_values("f2_fake", ascending=True)

fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=plot_emb, x="f2_fake", y="label", hue="representation", dodge=False, ax=ax)
ax.set_xlabel("F2 fake (test)")
ax.set_ylabel("")
ax.set_title("Embeddings estáticos: GloVe vs Word2Vec")
ax.legend(title="Representación", loc="lower right")
save_figure(fig, RESULTS_FIGURES / f"{FIGURE_PREFIX}embedding_comparison.png")
plt.show()

## 3. Parte B — Fine-tuning DistilBERT

Grilla de hiperparámetros en validación (F2 fake); **test se evalúa una sola vez** con la mejor configuración. Con `NLP_DEV_MODE=1` se submuestrea solo el train (10%) para acelerar iteraciones locales; la validación permanece completa.

Para entrenamiento completo con GPU: subir el proyecto a Colab, activar runtime GPU y ejecutar con `NLP_DEV_MODE` desactivado.

In [ ]:
SAMPLE_FRAC = 0.1 if DEV_MODE else 1.0

if DEV_MODE:
    LEARNING_RATES = [2e-5, 3e-5]
    BATCH_SIZES = [8]
    EPOCHS_LIST = [2]
    WARMUP_RATIOS = [0.0, 0.1]
    MAX_LENGTHS = [256]
    MAX_COMBOS = 4
else:
    LEARNING_RATES = [2e-5, 3e-5, 5e-5]
    BATCH_SIZES = [8, 16]
    EPOCHS_LIST = [2, 3]
    WARMUP_RATIOS = [0.0, 0.1]
    MAX_LENGTHS = [256]
    MAX_COMBOS = None

CHECKPOINT_DIR = RESULTS_MODELS / "distilbert_checkpoints"
EARLY_STOPPING_PATIENCE = 1

# Guarda DEV: en modo desarrollo (train al 10%, grilla recortada) los artefactos se
# escriben con sufijo _dev y NO sobrescriben los oficiales. La tabla comparativa final
# solo es válida con SAMPLE_FRAC=1.0 (NLP_DEV_MODE desactivado); además, consolidate_results
# descarta cualquier fila con sample_frac<1.0 como red de seguridad adicional.
DEV_SUFFIX = "" if SAMPLE_FRAC == 1.0 else "_dev"
TRANSFORMER_VAL_SEARCH_PATH = RESULTS_METRICS / f"transformer_val_search{DEV_SUFFIX}.csv"
TRANSFORMER_RESULTS_PATH = RESULTS_METRICS / f"transformer_results{DEV_SUFFIX}.csv"
BEST_DISTILBERT_DIR = RESULTS_MODELS / f"best_distilbert{DEV_SUFFIX}"

if DEV_MODE:
    print("DEV_MODE activo: SAMPLE_FRAC=0.1 (solo train); artefactos con sufijo _dev.")
print(f"SAMPLE_FRAC={SAMPLE_FRAC}")
print(f"GPU disponible: {torch.cuda.is_available()}")

In [ ]:
def compute_transformer_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    prec, rec, f1, _ = precision_recall_fscore_support(
        labels, preds, average=None, zero_division=0,
    )
    f2 = fbeta_score(labels, preds, beta=2, pos_label=1, zero_division=0)
    return {
        "f2_fake": f2,
        "f1_fake": f1[1] if len(f1) > 1 else 0,
        "accuracy": accuracy_score(labels, preds),
    }

In [ ]:
tr = pol_train.copy()
va = pol_val.copy()
te = pol_test.copy()

if SAMPLE_FRAC < 1.0:
    tr = tr.sample(frac=SAMPLE_FRAC, random_state=RANDOM_STATE)
    print(f"Muestra reducida (solo train): {len(tr):,} / {len(pol_train):,}")

tr_texts = prepare_transformer_texts(
    embedding_text_series(tr, TEXT_COL, normalize_source=normalize_source)
)
va_texts = prepare_transformer_texts(
    embedding_text_series(va, TEXT_COL, normalize_source=normalize_source)
)
te_texts = prepare_transformer_texts(
    embedding_text_series(te, TEXT_COL, normalize_source=normalize_source)
)

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
use_fp16 = torch.cuda.is_available()
combo_grid = list(
    product(LEARNING_RATES, BATCH_SIZES, EPOCHS_LIST, WARMUP_RATIOS, MAX_LENGTHS)
)
if MAX_COMBOS is not None:
    combo_grid = combo_grid[:MAX_COMBOS]

val_rows = []
best_val_f2 = -1.0
best_bundle = None

for lr, bs, epochs, warmup_ratio, max_len in combo_grid:
    print(f"\n=== lr={lr}, bs={bs}, epochs={epochs}, warmup={warmup_ratio}, max_len={max_len} ===")
    train_ds = NewsDataset(tr_texts, tr["label"], tokenizer, max_len)
    val_ds = NewsDataset(va_texts, va["label"], tokenizer, max_len)

    run_dir = CHECKPOINT_DIR / (
        f"lr{lr}_bs{bs}_ep{epochs}_wu{warmup_ratio}_len{max_len}"
    )
    model = AutoModelForSequenceClassification.from_pretrained(
        "distilbert-base-uncased", num_labels=2
    )
    args = TrainingArguments(
        output_dir=str(run_dir),
        learning_rate=lr,
        per_device_train_batch_size=bs,
        per_device_eval_batch_size=bs * 2,
        num_train_epochs=epochs,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f2_fake",
        greater_is_better=True,
        warmup_ratio=warmup_ratio,
        lr_scheduler_type="linear",
        logging_steps=50,
        seed=RANDOM_STATE,
        report_to="none",
        dataloader_num_workers=2,
        fp16=use_fp16,
    )
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        data_collator=data_collator,
        compute_metrics=compute_transformer_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)],
    )
    trainer.train()

    val_pred = trainer.predict(val_ds)
    y_val_pred = np.argmax(val_pred.predictions, axis=-1)
    val_metrics = compute_metrics(va["label"], y_val_pred)
    val_row = {
        "learning_rate": lr,
        "batch_size": bs,
        "epochs": epochs,
        "warmup_ratio": warmup_ratio,
        "max_length": max_len,
        "sample_frac": SAMPLE_FRAC,
        **val_metrics,
    }
    val_rows.append(val_row)
    print(f"F2 val: {val_metrics['f2_fake']:.4f}")

    if val_metrics["f2_fake"] > best_val_f2:
        best_val_f2 = val_metrics["f2_fake"]
        best_bundle = {
            "trainer": trainer,
            "model": trainer.model,
            "max_len": max_len,
            "lr": lr,
            "bs": bs,
            "epochs": epochs,
            "warmup_ratio": warmup_ratio,
        }

val_search_df = pd.DataFrame(val_rows)
val_search_df.to_csv(TRANSFORMER_VAL_SEARCH_PATH, index=False)
val_search_df.sort_values("f2_fake", ascending=False)

In [ ]:
test_ds = NewsDataset(te_texts, te["label"], tokenizer, best_bundle["max_len"])
pred = Trainer(model=best_bundle["model"], data_collator=data_collator).predict(test_ds)
y_pred = np.argmax(pred.predictions, axis=-1)
y_proba = torch.softmax(torch.tensor(pred.predictions), dim=-1)[:, 1].numpy()
test_m = compute_metrics(te["label"], y_pred, y_proba)

transformer_row = metrics_to_row(
    test_m,
    {
        "model": "distilbert-base-uncased",
        "learning_rate": best_bundle["lr"],
        "batch_size": best_bundle["bs"],
        "epochs": best_bundle["epochs"],
        "warmup_ratio": best_bundle["warmup_ratio"],
        "max_length": best_bundle["max_len"],
        "sample_frac": SAMPLE_FRAC,
        "dataset_scope": "politics",
        "split": "test",
        "use_source_normalization": normalize_source,
    },
)
transformer_df = pd.DataFrame([transformer_row])
transformer_df.to_csv(TRANSFORMER_RESULTS_PATH, index=False)
best_bundle["model"].save_pretrained(str(BEST_DISTILBERT_DIR))
tokenizer.save_pretrained(str(BEST_DISTILBERT_DIR))

print("Mejor config (val):", {
    "lr": best_bundle["lr"],
    "bs": best_bundle["bs"],
    "epochs": best_bundle["epochs"],
    "warmup_ratio": best_bundle["warmup_ratio"],
})
print("Métricas test:", {k: round(v, 4) for k, v in test_m.items() if k != "roc_auc"})
if SAMPLE_FRAC < 1.0:
    print(
        f"\n⚠️  DEV_MODE (SAMPLE_FRAC={SAMPLE_FRAC}): se escribió {TRANSFORMER_RESULTS_PATH.name} "
        f"y {BEST_DISTILBERT_DIR.name}; NO se tocaron los artefactos oficiales. "
        "Re-correr con NLP_DEV_MODE desactivado para la tabla comparativa final."
    )
transformer_df

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
plot_val = val_search_df.sort_values("f2_fake", ascending=False)
plot_val["config"] = plot_val.apply(
    lambda r: f"lr={r['learning_rate']}, bs={int(r['batch_size'])}, wu={r['warmup_ratio']}",
    axis=1,
)
sns.barplot(data=plot_val, x="f2_fake", y="config", ax=ax, color="#4c72b0")
ax.set_xlabel("F2 fake (validación)")
ax.set_ylabel("")
ax.set_title("DistilBERT: búsqueda de hiperparámetros")
save_figure(fig, RESULTS_FIGURES / f"{FIGURE_PREFIX}transformer_hp_search.png")
plt.show()

## 4. Consolidación y comparación global

Unir resultados de embeddings y transformer con experimentos previos en `all_model_results.csv`.

In [ ]:
all_results = consolidate_results()
exp3_mask = pd.Series(False, index=all_results.index)
if "representation" in all_results.columns:
    exp3_mask |= all_results["representation"].isin(["glove_avg", "word2vec_avg"])
if "model" in all_results.columns:
    exp3_mask |= all_results["model"] == "distilbert-base-uncased"
exp3_results = all_results[exp3_mask].copy()

if not exp3_results.empty:
    exp3_results["label"] = np.where(
        exp3_results["model"] == "distilbert-base-uncased",
        "DistilBERT",
        exp3_results["representation"] + " / " + exp3_results["model"],
    )
    plot_all = exp3_results.sort_values("f2_fake", ascending=True)
    fig, ax = plt.subplots(figsize=(9, 5))
    sns.barplot(data=plot_all, x="f2_fake", y="label", ax=ax, color="#55a868")
    ax.set_xlabel("F2 fake (test)")
    ax.set_ylabel("")
    ax.set_title("Experimento 3: embeddings vs DistilBERT (politics, test)")
    save_figure(fig, RESULTS_FIGURES / f"{FIGURE_PREFIX}exp3_overview.png")
    plt.show()

all_results.head(10)

## Conclusiones

- **GloVe** aporta vectores genéricos de dominio periodístico; **Word2Vec** entrenado en el corpus evalúa si el vocabulario 2015–2017 mejora la separación fake/real.
- **DistilBERT** captura contexto intra-oración; la grilla en validación selecciona LR, batch, épocas y warmup antes de evaluar test una vez.
- BERT-base queda documentado como fuera de alcance en CPU; Colab con GPU es la vía opcional para extender el experimento.
- Ningún modelo verifica hechos; aprenden patrones del dataset.